In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

base_path = "/home/bingxing2/ailab/scxlab0055/project/04_Inversion/ADFWI-github/examples/DR-FWI/multi-parameters/ISO_elastic/Anomaly-vp_vs_rho-shotTop-recTop-v1/data"

init_model = np.load(os.path.join(base_path,"model/init_model.npz"))
true_model = np.load(os.path.join(base_path,"model/true_model.npz"))
init_vp = init_model["vp"]
init_vs = init_model["vs"]
init_rho = init_model["rho"]
true_vp = true_model["vp"]
true_vs = true_model["vs"]
true_rho = true_model["rho"]

ox, oz = 0, 0             # Origin coordinates for x and z directions
nz, nx = 100, 200         # Grid dimensions in z and x directions
dx, dz = 30, 30           # Grid spacing in x and z directions
nt, dt = 2500, 0.0025     # Time steps and time interval
nabc = 50                 # Thickness of the absorbing boundary layer
f0 = 5                    # Initial frequency in Hz
free_surface = True       # Enable free surface boundary condition
       
x = np.arange(nx)*dx/1000
z = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)
src_z = np.array([2 for i in range(2, nx-1, 5)])*dz/1000  # Z-coordinates for sources
src_x = np.array([i for i in range(2, nx-1, 5)])*dz/1000  # X-coordinates for sources
rcv_z = np.array([2 for i in range(0, nx, 1)])*dz/1000  # Z-coordinates for receivers
rcv_x = np.array([j for j in range(0, nx, 1)])*dz/1000  # X-coordinates for receivers
vpmin = true_vp.min()
vpmax = true_vp.max()
vsmin = true_vs.min()
vsmax = true_vs.max()
rhomin = true_rho.min()
rhomax = true_rho.max()

MAX_ITER = 1000

In [ ]:
itervp_baseline         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-baseline-v1/iter_vp.npz"))["data"][:370]
itervs_baseline         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-baseline-v1/iter_vs.npz"))["data"][:370]
iterrho_baseline        = np.load(os.path.join(base_path,"inversion-vp_vs_rho-baseline-v1/iter_rho.npz"))["data"][:370]

itervp_CNN_2X4          = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x4/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2X8          = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x8/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2X16         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x16/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2X32         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x32/iter_vp.npz"))["data"][:MAX_ITER]

itervs_CNN_2X4          = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x4/iter_vs.npz"))["data"][:MAX_ITER]
itervs_CNN_2X8          = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x8/iter_vs.npz"))["data"][:MAX_ITER]
itervs_CNN_2X16         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x16/iter_vs.npz"))["data"][:MAX_ITER]
itervs_CNN_2X32         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x32/iter_vs.npz"))["data"][:MAX_ITER]

iterrho_CNN_2X4         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x4/iter_rho.npz"))["data"][:MAX_ITER]
iterrho_CNN_2X8         = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x8/iter_rho.npz"))["data"][:MAX_ITER]
iterrho_CNN_2X16        = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x16/iter_rho.npz"))["data"][:MAX_ITER]
iterrho_CNN_2X32        = np.load(os.path.join(base_path,"inversion-vp_vs_rho-CNN3-2x32/iter_rho.npz"))["data"][:MAX_ITER]

In [ ]:
from scipy.interpolate import griddata
import matplotlib.transforms as mtransforms
    
def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None,cmap='rainbow',text_color='w'):
    plt.rc('font',family='Times New Roman')
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)
    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 16)
    ax.set_title(title,fontsize=16)
    ax.text(0.1,0.45,MSE,fontsize=16,c=text_color)
    return plm

def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

import matplotlib as mpl
def add_bottom_cax(ax, pad, height,shrink=1):
    axpos = ax.get_position()
    width = axpos.x1 - axpos.x0
    left_position = axpos.x0 + width * (1 - shrink) / 2
    caxpos = mpl.transforms.Bbox.from_extents(
        left_position,
        axpos.y0 - pad,
        left_position + width * shrink,
        axpos.y0 - pad + height
    )
    cax = ax.figure.add_axes(caxpos)
    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 16)
    if not show_xlabel:
        ax.set_xticks([])
    # else:
    #     ax.tick_params(labelsize = 15)
        
    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 12)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=16)

In [ ]:
from ADFWI.utils.assessment_metric import MAPE, MSE

from matplotlib.ticker import MaxNLocator
plt.rcParams['svg.fonttype'] = 'none'

fig,axs = plt.subplots(3,3,figsize=(14,7))
im1 = plot_vel_single_for_all(fig,axs[0][0],true_vp  ,MSE="(a)",vmin=vpmin,vmax=vpmax,cmap='jet')
im2 = plot_vel_single_for_all(fig,axs[0][1],true_vs  ,MSE="(b)",vmin=vsmin,vmax=vsmax,cmap='jet')
im3 = plot_vel_single_for_all(fig,axs[0][2],true_rho ,MSE="(c)",vmin=rhomin,vmax=rhomax,cmap='jet')

plot_vel_single_for_all(fig,axs[1][0],itervp_baseline[-1] ,MSE="(d) MAPE:{:.2f}".format(MAPE(true_vp ,itervp_baseline[-1])),vmin=vpmin,vmax=vpmax,cmap='jet')
plot_vel_single_for_all(fig,axs[1][1],itervs_baseline[-1] ,MSE="(e) MAPE:{:.2f}".format(MAPE(true_vs ,itervs_baseline[-1])),vmin=vsmin,vmax=vsmax,cmap='jet')
plot_vel_single_for_all(fig,axs[1][2],iterrho_baseline[-1],MSE="(f) MAPE:{:.2f}".format(MAPE(true_rho,iterrho_baseline[-1])),vmin=rhomin,vmax=rhomax,cmap='jet')

plot_vel_single_for_all(fig,axs[2][0],itervp_CNN_2X32[-1] ,MSE="(g) MAPE:{:.2f}".format(MAPE(true_vp ,itervp_CNN_2X32[-1])),vmin=vpmin,vmax=vpmax,cmap='jet')
plot_vel_single_for_all(fig,axs[2][1],itervs_CNN_2X32[-1] ,MSE="(h) MAPE:{:.2f}".format(MAPE(true_vs ,itervs_CNN_2X32[-1])),vmin=vsmin,vmax=vsmax,cmap='jet')
plot_vel_single_for_all(fig,axs[2][2],iterrho_CNN_2X32[-1],MSE="(i) MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN_2X32[-1])),vmin=rhomin,vmax=rhomax,cmap='jet')

axs[0][0].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][0].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][1].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][1].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][2].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][2].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)

axs[0][0].set_ylabel("True", fontsize=15)
axs[1][0].set_ylabel("Traditional FWI", fontsize=15)
axs[2][0].set_ylabel(r"CNN-$v_p$", fontsize=15)

# Move ticks and labels to the right side for axs[0][1]
axs[0][2].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[0][2].yaxis.set_ticks_position('right')
axs[0][2].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[0][2].yaxis.set_label_position("right")

axs[1][2].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[1][2].yaxis.set_ticks_position('right')
axs[1][2].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[1][2].yaxis.set_label_position("right")

axs[2][2].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[2][2].yaxis.set_ticks_position('right')
axs[2][2].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[2][2].yaxis.set_label_position("right")

# set the ticks number
axs[0][2].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[1][2].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[2][2].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[2][0].xaxis.set_major_locator(MaxNLocator(nbins=5))
axs[2][1].xaxis.set_major_locator(MaxNLocator(nbins=5))
axs[2][2].xaxis.set_major_locator(MaxNLocator(nbins=5))

axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[0][2].set_xticks([])
axs[1][0].set_xticks([])
axs[1][1].set_xticks([])
axs[1][2].set_xticks([])

axs[0][1].set_yticks([])
axs[0][0].set_yticks([])
axs[1][1].set_yticks([])
axs[1][0].set_yticks([])
axs[2][1].set_yticks([])
axs[2][0].set_yticks([])

axs[2][0].set_xlabel("Distance (km)", fontsize=15)
axs[2][1].set_xlabel("Distance (km)", fontsize=15)
axs[2][2].set_xlabel("Distance (km)", fontsize=15)

# Set titles for each subplot
axs[0][0].set_title(r"P-wave Velocity", fontsize=15)
axs[0][1].set_title(r"S-wave Velocity", fontsize=15)
axs[0][2].set_title(r"Density"        , fontsize=15)

# Colorbars on the right side
cax1 = add_bottom_cax(axs[2][0], pad=0.125, height=0.025, shrink=0.95)
cbar1 = fig.colorbar(im1, cax=cax1, orientation='horizontal')
cbar1.ax.tick_params(labelsize=12)
cbar1.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar1.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cax2 = add_bottom_cax(axs[2][1], pad=0.125, height=0.025, shrink=0.95)
cbar2 = fig.colorbar(im2, cax=cax2, orientation='horizontal')
cbar2.ax.tick_params(labelsize=12)
cbar2.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar2.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cax3 = add_bottom_cax(axs[2][2], pad=0.125, height=0.025, shrink=0.95)
cbar3 = fig.colorbar(im3, cax=cax3, orientation='horizontal')
cbar3.ax.tick_params(labelsize=12)
cbar3.ax.text(1.02, 0.5, r'$kg/m^3$', fontsize=14, transform=cbar3.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

plt.subplots_adjust(hspace=0.1,wspace=0.05)

# plt.savefig("./Figures/Figure8_MultiParameter-AnomalyModel.png",bbox_inches='tight',dpi=300)
# plt.savefig("./Figures_PDF/Figure8_MultiParameter-AnomalyModel.pdf",bbox_inches='tight',dpi=300,format="pdf")
plt.savefig("./Figures_SVG/Figure8_MultiParameter-AnomalyModel.svg",bbox_inches='tight',format="svg")
plt.show()